In [25]:
import pandas as pd
import json
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [2]:
movies = pd.read_csv("dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("dataset/tmdb_5000_credits.csv")

In [3]:
merged_df = movies.merge(credits, left_on="id", right_on="movie_id")

In [4]:
merged_df = merged_df[["genres", "keywords", "overview", "crew", "cast", "title_x", "movie_id"]]
merged_df = merged_df.rename(columns={"title_x": "title"})

In [5]:
merged_df = merged_df.dropna()

In [6]:
merged_df["genres"] = merged_df["genres"].apply(lambda x: [i["name"].replace(" ", "") for i in json.loads(x)])

In [7]:
merged_df["keywords"] = merged_df["keywords"].apply(lambda x: [i["name"].replace(" ", "") for i in json.loads(x)])

In [8]:
merged_df["cast"] = merged_df["cast"].apply(lambda x: [i["name"].replace(" ", "") for i in json.loads(x)][:3])

In [9]:
merged_df["crew"] = merged_df["crew"].apply(lambda x: [i["name"].replace(" ", "") for i in json.loads(x) if i["job"]=="Director"][:1])

In [11]:
merged_df["overview"] = merged_df["overview"].apply(lambda x: x.split())

In [12]:
merged_df["tags"] = merged_df["overview"]+merged_df["genres"]+merged_df["keywords"]+merged_df["cast"]+merged_df["crew"]

In [13]:
merged_df = merged_df[["movie_id", "title", "tags"]]

In [15]:
merged_df["tags"] = merged_df["tags"].apply(lambda x: " ".join(x).lower())

In [19]:
docs = nlp.pipe(merged_df["tags"], batch_size=50)
merged_df["tags"] = [[token.lemma_ for token in doc if not token.is_punct and not token.is_stop] for doc in docs]

In [22]:
cv = CountVectorizer(tokenizer=lambda x: x, lowercase=False, max_features=5000)
vectors = cv.fit_transform(merged_df["tags"])

/home/vikas/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [23]:
similarity = cosine_similarity(vectors)

In [26]:
def recommend(movie, n=5):
    movie_index = np.where(merged_df["title"]==movie)[0][0]
    distances = similarity[movie_index]
    movies_list = sorted(enumerate(distances), reverse=True, key=lambda x: x[1])[1:n+1]
    for index, score in movies_list:
        print(merged_df.iloc[index].title)

In [28]:
recommend("Batman Begins")

The Dark Knight
The Dark Knight Rises
Batman & Robin
Batman v Superman: Dawn of Justice
Batman


In [29]:
recommend("Avatar")

Falcon Rising
Aliens
Independence Day
Titan A.E.
Small Soldiers


In [30]:
recommend("Harry Potter and the Philosopher's Stone")

Harry Potter and the Chamber of Secrets
Harry Potter and the Half-Blood Prince
Harry Potter and the Goblet of Fire
Harry Potter and the Prisoner of Azkaban
Harry Potter and the Order of the Phoenix


In [31]:
recommend("Spider-Man")

Spider-Man 3
Spider-Man 2
The Amazing Spider-Man
The Amazing Spider-Man 2
Arachnophobia


In [32]:
recommend("Superman")

Superman II
Superman Returns
Superman IV: The Quest for Peace
Superman III
Man of Steel


In [33]:
recommend("The Avengers")

Avengers: Age of Ultron
Iron Man 3
Captain America: The First Avenger
Iron Man
Iron Man 2


In [34]:
recommend("Pirates of the Caribbean: At World's End")

Pirates of the Caribbean: Dead Man's Chest
Pirates of the Caribbean: The Curse of the Black Pearl
Pirates of the Caribbean: On Stranger Tides
VeggieTales: The Pirates Who Don't Do Anything
Silver Medalist
